In [1]:
import torch
import torch.nn as nn
import numpy as np
import requests
import re
import math

# ==========================================
# 0. SETUP & DEVICE
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# ==========================================
# 1. DATASET: ALICE IN WONDERLAND (Simple English)
# ==========================================
print("Downloading 'Alice in Wonderland' dataset...")

# Using a reliable GitHub gist mirror to avoid Project Gutenberg blocking Colab
url = "https://gist.githubusercontent.com/phillipj/4944029/raw/75ba2243dd5ec2875f629bf5d79f6c1e4b5a8b46/alice_in_wonderland.txt"

# Add a User-Agent header to behave more like a standard web browser
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
text = response.text

# Find the start of the actual story
start_idx = text.find("CHAPTER I")
if start_idx == -1:
    start_idx = 0 # Fallback in case the exact string varies

# Grab the first 50,000 characters
clean_text = text[start_idx:start_idx + 100000].lower()

# Keep only basic letters, numbers, and basic punctuation
clean_text = re.sub(r'[^a-z0-9\s.,!?]', '', clean_text)

# Normalize spaces
clean_text = re.sub(r'\s+', ' ', clean_text).strip()

print(f"Dataset ready! Total characters: {len(clean_text)}")
print("="*60)

Using device: cpu

Dataset ready! Total characters: 93300


In [2]:
# COMPONENT I: LSTM (CHARACTER-LEVEL)
# ==========================================
print("\n--- Component I: LSTM (Character-Level) ---")

# 1. Tokenization
chars = sorted(list(set(clean_text)))
char_to_int = {c: i for i, c in enumerate(chars)}
int_to_char = {i: c for i, c in enumerate(chars)}
char_vocab_size = len(chars)

# Batch Generator for Characters
def get_char_batch(data_text, seq_len, batch_size):
    indices = np.random.randint(0, len(data_text) - seq_len - 1, batch_size)
    x = torch.tensor([[char_to_int[c] for c in data_text[i:i+seq_len]] for i in indices], dtype=torch.long)
    y = torch.tensor([char_to_int[data_text[i+seq_len]] for i in indices], dtype=torch.long)
    return x.to(device), y.to(device)

# 2. Model Architecture
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, num_layers=2) # 2 layers for better learning
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]) # Output of the last time step

# 3. Training setup
lstm_seq_len = 30
lstm_model = CharLSTM(char_vocab_size, embed_dim=64, hidden_dim=128).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_lstm = torch.optim.Adam(lstm_model.parameters(), lr=0.005)

# Train
epochs_lstm = 5000
lstm_model.train()
for epoch in range(epochs_lstm):
    X_batch, y_batch = get_char_batch(clean_text, lstm_seq_len, batch_size=128)

    optimizer_lstm.zero_grad()
    outputs = lstm_model(X_batch)
    loss = criterion(outputs, y_batch)
    loss.backward()
    optimizer_lstm.step()

    if (epoch+1) % 300 == 0:
        print(f"LSTM Step [{epoch+1}/{epochs_lstm}], Loss: {loss.item():.4f}")

# 4. Generation
lstm_model.eval()
seed_text = "alice looked down the rabbit " # Must be close to seq_len
generated_char = seed_text
current_seq = torch.tensor([[char_to_int[c] for c in seed_text[-lstm_seq_len:]]], dtype=torch.long).to(device)

print("\nGenerating Text...")
with torch.no_grad():
    for _ in range(600): # Generate 100 characters
        output = lstm_model(current_seq)

        # Adding a bit of "temperature" to make predictions more natural
        probs = torch.softmax(output / 0.8, dim=-1).cpu().numpy().flatten()
        next_char_idx = np.random.choice(char_vocab_size, p=probs)

        generated_char += int_to_char[next_char_idx]
        current_seq = torch.tensor([[char_to_int[c] for c in generated_char[-lstm_seq_len:]]], dtype=torch.long).to(device)

print(f"\nLSTM Output:\n{generated_char}\n")
print("="*60)


--- Component I: LSTM (Character-Level) ---
LSTM Step [300/5000], Loss: 1.7070
LSTM Step [600/5000], Loss: 1.7100
LSTM Step [900/5000], Loss: 1.4878
LSTM Step [1200/5000], Loss: 1.2980
LSTM Step [1500/5000], Loss: 1.2510
LSTM Step [1800/5000], Loss: 1.2579
LSTM Step [2100/5000], Loss: 1.2939
LSTM Step [2400/5000], Loss: 1.4184
LSTM Step [2700/5000], Loss: 1.1451
LSTM Step [3000/5000], Loss: 1.1567
LSTM Step [3300/5000], Loss: 1.2739
LSTM Step [3600/5000], Loss: 1.2986
LSTM Step [3900/5000], Loss: 0.9236
LSTM Step [4200/5000], Loss: 1.1598
LSTM Step [4500/5000], Loss: 1.1705
LSTM Step [4800/5000], Loss: 1.1617

Generating Text...

LSTM Output:
alice looked down the rabbit small be the dormouse was a little shool down againstly. im not all rous fight in the minute or two a moment than in those of the queen the hatter. his all some way to say and those gardeners as she well as i dont butll was a long air, she would not this side to the dormouse again, as he confusions as the game again, 

In [3]:

# COMPONENT II: TRANSFORMER (WORD-LEVEL)
# ==========================================
print("\n--- Component II: Transformer (Word-Level) ---")

# 1. Word-level Tokenization
# Pad punctuation so words and punctuation are treated separately
word_text = clean_text.replace('.', ' . ').replace(',', ' , ').replace('!', ' ! ').replace('?', ' ? ')
words = word_text.split()

unique_words = sorted(list(set(words)))
word_to_int = {w: i for i, w in enumerate(unique_words)}
int_to_word = {i: w for i, w in enumerate(unique_words)}
word_vocab_size = len(unique_words)

# Batch Generator for Words
def get_word_batch(word_list, seq_len, batch_size):
    indices = np.random.randint(0, len(word_list) - seq_len - 1, batch_size)
    x = torch.tensor([[word_to_int[w] for w in word_list[i:i+seq_len]] for i in indices], dtype=torch.long)
    y = torch.tensor([word_to_int[word_list[i+seq_len]] for i in indices], dtype=torch.long)
    return x.to(device), y.to(device)

# 2. Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# 3. Model Architecture
class WordTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, x):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        out = self.transformer_encoder(x)
        return self.fc(out[:, -1, :])

# 4. Training Setup
word_seq_len = 5
transformer_model = WordTransformer(vocab_size=word_vocab_size, d_model=128, nhead=4, num_layers=2).to(device)
criterion_tf = nn.CrossEntropyLoss()
optimizer_tf = torch.optim.Adam(transformer_model.parameters(), lr=0.001)

# Train
epochs_tf = 5000
transformer_model.train()
for epoch in range(epochs_tf):
    X_batch, y_batch = get_word_batch(words, word_seq_len, batch_size=128)

    optimizer_tf.zero_grad()
    outputs = transformer_model(X_batch)
    loss = criterion_tf(outputs, y_batch)
    loss.backward()
    optimizer_tf.step()

    if (epoch+1) % 300 == 0:
        print(f"Transformer Step [{epoch+1}/{epochs_tf}], Loss: {loss.item():.4f}")

# 5. Generation
transformer_model.eval()
seed_words = ["alice", "was", "beginning", "to", "get"]
generated_words = list(seed_words)
current_word_seq = torch.tensor([[word_to_int[w] for w in seed_words]], dtype=torch.long).to(device)

print("\nGenerating Text...")
with torch.no_grad():
    for _ in range(30): # Generate 30 words
        output = transformer_model(current_word_seq)

        # Temperature sampling for words
        probs = torch.softmax(output / 0.8, dim=-1).cpu().numpy().flatten()
        next_word_idx = np.random.choice(word_vocab_size, p=probs)
        next_word = int_to_word[next_word_idx]

        generated_words.append(next_word)
        current_word_seq = torch.tensor([[word_to_int[w] for w in generated_words[-word_seq_len:]]], dtype=torch.long).to(device)

# Clean up spacing around punctuation for output
final_tf_output = ' '.join(generated_words).replace(' .', '.').replace(' ,', ',').replace(' !', '!').replace(' ?', '?')
print(f"\nTransformer Output:\n{final_tf_output}\n")


--- Component II: Transformer (Word-Level) ---
Transformer Step [300/5000], Loss: 4.6403
Transformer Step [600/5000], Loss: 4.1946
Transformer Step [900/5000], Loss: 3.2933
Transformer Step [1200/5000], Loss: 2.7329
Transformer Step [1500/5000], Loss: 2.3138
Transformer Step [1800/5000], Loss: 1.9330
Transformer Step [2100/5000], Loss: 1.3497
Transformer Step [2400/5000], Loss: 1.3786
Transformer Step [2700/5000], Loss: 1.3748
Transformer Step [3000/5000], Loss: 1.1242
Transformer Step [3300/5000], Loss: 0.8492
Transformer Step [3600/5000], Loss: 0.8735
Transformer Step [3900/5000], Loss: 0.7625
Transformer Step [4200/5000], Loss: 0.5120
Transformer Step [4500/5000], Loss: 0.7832
Transformer Step [4800/5000], Loss: 0.6704

Generating Text...

Transformer Output:
alice was beginning to get very tired of sitting, fast asleep, and the other two were using it as a cushion, resting their elbows on it, and talking over its head

